### task-pair LoRA fine-tune — `Llama-3.1-8B-Instruct`

story in, two RG lines out. loss on the answer tokens only.
this is v2: v1 trained on bare grammar text and it didn't teach translation (see `RESULTS.md`).

training runs on a Modal A100 — the cells below launch it and stream logs back here.
swap the model by changing `TRAIN_MODEL_ID` in the next cell and re-running the notebook.

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# these are read at import time, so set them BEFORE importing train_pairs
os.environ["TRAIN_MODEL_ID"] = "meta-llama/Llama-3.1-8B-Instruct"
os.environ["TRAIN_GPU"] = "A100-80GB"

In [ ]:
import sys
sys.path.insert(0, "training")
import train_pairs

train_pairs.MODEL_ID, train_pairs.TRAIN_GPU

In [ ]:
train_pairs.PRESETS

### config

r=16, all layers, lr 2e-4 cosine, 3 epochs. same recipe as v1 so the two are comparable.

In [ ]:
PRESET = "v2-8b"      # "smoke-v2" for a 50-step trial first

cfg, repo = train_pairs.load_local()
config = train_pairs.build_config(PRESET, seed=0, samples=0, epochs=0,
                                  max_steps=0, run_name="")
config

### the data, as the model will see it

worth eyeballing once: check the prompt ends at the generation turn and the
answer tokens are the only ones with a loss.

In [ ]:
rows = train_pairs.load_rows(cfg.ftc.PATHS["train_v2"] / "train.jsonl")
hold = train_pairs.load_rows(cfg.ftc.PATHS["train_v2"] / "holdout.jsonl")
payload = train_pairs.build_payload(rows, repo, train_pairs.MODEL_ID)
len(payload)

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(train_pairs.MODEL_ID)
ck = train_pairs.chat_kwargs_for(train_pairs.MODEL_ID)
ds, dropped, lengths = train_pairs.build_examples(tok, payload[:20], ck, 2048)
train_pairs.seq_stats(lengths), dropped

In [ ]:
ex = ds[0]
n_masked = sum(1 for l in ex["labels"] if l == -100)
print("prompt tokens (no loss):", n_masked)
print("answer tokens (loss):  ", len(ex["labels"]) - n_masked)

print("\n--- what the loss is computed on ---")
print(tok.decode([l for l in ex["labels"] if l != -100]))

In [ ]:
# tail of the prompt, to confirm the chat template ended where it should
print(tok.decode(ex["input_ids"][:n_masked])[-400:])

### train

~65 min for the 8B on an A100. logs stream into the cell.
checkpoints land on the Modal volume at step 0 and every 100 steps.

In [ ]:
with train_pairs.app.run():
    record = train_pairs.train.remote(payload,
                                      train_pairs.build_payload(hold, repo, train_pairs.MODEL_ID),
                                      config)

record["steps"], record["train_loss_final"], record["train_seconds"]

In [ ]:
import matplotlib.pyplot as plt

curve = record["loss_curve"]
plt.figure(figsize=(7, 3))
plt.plot([p["step"] for p in curve], [p["loss"] for p in curve], lw=1)
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss"); plt.show()

### did it actually learn

held-out pairs it never trained on. base vs adapter, then a greedy sample.

In [ ]:
p = record["probes"]
print(f"holdout loss   {p['holdout_completion_loss_base']:.3f} -> {p['holdout_completion_loss_ft']:.3f}")
print(f"perplexity     {p['holdout_completion_ppl_base']:.2f} -> {p['holdout_completion_ppl_ft']:.2f}")

In [ ]:
print("--- reference ---");   print(p["generation_reference"])
print("\n--- base model ---"); print(p["generation_base"][:300])
print("\n--- fine-tuned ---"); print(p["generation_ft"])

In [ ]:
# does the fine-tuned answer actually grade correct?
sys.path.insert(0, str(cfg.ftc.REPO))
from checkform import grade

h0 = hold[0]
grade(p["generation_ft"], {"canonical_e": h0["canonical_e"],
                            "canonical_f": h0["canonical_f"]})

### next

run the evals on the saved adapter — `eval/run-evals.ipynb`.
the run record is written to `runs/ft-v2/train-<run_name>.json`.